# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UsmanRizwan20/week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane as an ML task (type)

My lane is **CTR / Engagement Opportunity Scoring**.

I will frame this as a **ranking/scoring** problem. The goal is to assign an opportunity score to content pages and rank them based on which pages should be reviewed first for potential CTR or engagement improvement.

This is a ranking problem because the content team has limited time and needs a prioritized list rather than a simple yes/no decision. The output should help the team identify which visible pages are the strongest candidates for review.

## 2. Target or proxy

The target will be a **CTR opportunity score**, used as a proxy for how strongly a page should be prioritized for review.

The starter dataset does not contain a directly observed future outcome such as "CTR improved after an intervention." Therefore, I will not claim that the score predicts actual future CTR improvement.

Instead, the proxy will be based on observable signals such as search impressions, clicks, CTR, average position, and engagement signals. Pages with meaningful search visibility but relatively weak click or engagement performance can represent stronger review opportunities.

This is therefore a **decision-support ranking**, not a causal prediction of how much CTR will increase after an action.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

The main success metric will be **Precision@K**, for example Precision@50.

Precision@50 measures how many of the 50 highest-ranked pages are actually considered relevant opportunities according to the observed opportunity definition.

This metric fits the real decision because the content team cannot review every page. They need a short, prioritized list, so the quality of the top-ranked pages matters more than the performance across every page.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The unit of analysis is **one content page**. Each row represents one pseudonymized content item, with its observed search and engagement measurements aggregated over the available 90-day window.

In [20]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Unit of analysis: One row represents one pseudonymized content page.")

lane_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update"
]

lane_df = df[lane_columns]

lane_df.head(10)

Dataset shape: (30000, 44)
Unit of analysis: One row represents one pseudonymized content page.


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,scroll_rate,content_age_days,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,17,5.88,4.55,187,20
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,9,0.00,10.00,445,25
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,11,0.00,28.57,141,20
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,78,1.28,3.45,463,22
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,145,0.00,24.29,263,14
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,5,0.00,25.00,147,20
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,1,0.00,0.00,90,20
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,28,3.57,7.14,445,22
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,68,5.88,6.25,90,20
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,3,0.00,0.00,257,104


### Target/proxy sketch

The `ctr_opportunity_score` column is shown as a placeholder to make the intended output concrete. Its final definition will be established later using only appropriate observable signals and a clearly defined decision window.

At this stage, the column is intentionally not presented as a measured future outcome.

In [21]:
# Sketch of the future opportunity-score column
target_sketch = lane_df[
    ["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position"]
].head(10).copy()

target_sketch["ctr_opportunity_score"] = None

target_sketch

,content_id,impressions_90d,clicks_90d,ctr,avg_position,ctr_opportunity_score
0,content_304f48230142,3803,29,0.76,10.6,None
1,content_a1fb4e703a9e,15320,7,0.05,20.3,None
2,content_9aa793d4d895,12581,11,0.09,36.5,None
3,content_331d6c4de07b,11751,58,0.49,6.2,None
4,content_d99b7a2d90ca,19140,24,0.13,44.0,None
5,content_d4084a4bc775,3970,1,0.03,8.5,None
6,content_9a34b442b552,20,0,0.00,7.0,None
7,content_a63219c6e95a,1724,1,0.06,21.2,None
8,content_5e6c160719bc,32574,29,0.09,46.0,None
9,content_c27558df2b0c,1240,2,0.16,4.9,None


## 5. Why ML beats a fixed rule here

A simple fixed rule could flag pages such as:

"if CTR < X%, review the page."

However, CTR alone does not capture the full opportunity. A page with very low CTR but only a small number of impressions may be less useful to review than a page with substantial search visibility and slightly weaker-than-expected click performance.

The opportunity also depends on multiple signals, including impressions, clicks, average position, sessions, engagement, content age, and freshness.

A fixed rule would require manually choosing thresholds for these signals and combining them into a hand-written formula. ML can potentially learn interactions between multiple observed signals and produce a more useful prioritization of pages.

The goal is therefore not to replace human review. The goal is to make the review queue more efficient by ranking pages that deserve attention first.

The result should be treated as **decision-support and directional**, not as causal proof that changing a page will increase CTR.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.